In [1]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

C:\Users\parsh\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
248,"Sorry, but Jacqueline Hyde (get it??? - Jack L...",negative
128,"I may not have the longest of attention-spans,...",negative
303,Soul Calibur is more solid than it ever was......,positive
376,I found the pace to be glacial and the origina...,negative
546,I watched this film in shire joy.<br /><br />T...,positive


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
248,sorry jacqueline hyde get it jack l hyde jekyl...,negative
128,may longest attention span second movie refuse...,negative
303,soul calibur solid ever wa new character creat...,positive
376,found pace glacial original story blown way pr...,negative
546,watched film shire joy br br this possibly one...,positive


In [5]:
df['sentiment'].value_counts()

sentiment
positive    252
negative    248
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
248,sorry jacqueline hyde get it jack l hyde jekyl...,0
128,may longest attention span second movie refuse...,0
303,soul calibur solid ever wa new character creat...,1
376,found pace glacial original story blown way pr...,0
546,watched film shire joy br br this possibly one...,1


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
vectorizer = CountVectorizer(max_features=150)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Parshaw3558/sentiment-analysis-mlops.mlflow')
dagshub.init(repo_owner='Parshaw3558', repo_name='sentiment-analysis-mlops', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


Accessing as Parshaw3558

Initialized MLflow to track repo "Parshaw3558/sentiment-analysis-mlops"

Repository Parshaw3558/sentiment-analysis-mlops initialized!

2026/06/21 11:56:29 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2de38ce6189a4e7a8605e67f013494c7', creation_time=1782023196484, experiment_id='0', last_update_time=1782023196484, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [13]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 150)
        mlflow.log_param("test_size", 0.3)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-21 11:59:53,424 - INFO - Starting MLflow run...
2026-06-21 11:59:56,232 - INFO - Logging preprocessing parameters...
2026-06-21 11:59:57,321 - INFO - Initializing Logistic Regression model...
2026-06-21 11:59:57,322 - INFO - Fitting the model...
2026-06-21 11:59:57,370 - INFO - Model training complete.
2026-06-21 11:59:57,371 - INFO - Logging model parameters...
2026-06-21 11:59:57,754 - INFO - Making predictions...
2026-06-21 11:59:57,758 - INFO - Calculating evaluation metrics...
2026-06-21 11:59:57,777 - INFO - Logging evaluation metrics...
2026-06-21 11:59:57,853 - WARNING - Retrying (Retry(total=6, connect=7, read=6, redirect=7, status=7)) after connection broken by 'RemoteDisconnected('Remote end closed connection without response')': /Parshaw3558/sentiment-analysis-mlops.mlflow/api/2.0/mlflow/runs/log-metric
2026-06-21 12:00:00,193 - INFO - Saving and logging the model...
2026/06/21 12:00:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` in

🏃 View run adorable-doe-212 at: https://dagshub.com/Parshaw3558/sentiment-analysis-mlops.mlflow/#/experiments/0/runs/c89c10a8d01c4002b77264361d3d90c1
🧪 View experiment at: https://dagshub.com/Parshaw3558/sentiment-analysis-mlops.mlflow/#/experiments/0
